<a href="https://colab.research.google.com/github/albhoe/593Project/blob/main/tagproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft

In [ ]:
from google.colab import drive
import pandas as pd
import torch
from transformers import BartForConditionalGeneration, BartTokenizer, BartForSequenceClassification
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
import io # Import io module for StringIO
import os
import gc

drive.mount('/content/drive')

line_count = 0
df = pd.DataFrame()

try:
    file_path_drive = '/content/drive/MyDrive/593project/ao3_14400001-14500000.jsonl'
    with open(file_path_drive, 'r') as f:
        for line in f:
            if line_count >= 20:
                break
            line_count += 1
            # Use io.StringIO to pass the literal JSON string safely
            df = pd.concat([df, pd.read_json(io.StringIO(line), lines=True)], ignore_index=True)

except Exception as e:
    print(f"An error occurred loading from Drive: {e}")
metadata_df = pd.json_normalize(df['metadata'])
df = df.drop(columns=['metadata','id'])
df = pd.concat([df.reset_index(drop=True), metadata_df.reset_index(drop=True)], axis=1)
df = df.drop(columns=['published','words','Collections','Series','Fandoms','Archive Warnings','Relationships','Character','Categories'])
print("\nFirst 5 rows of data from Google Drive:")
display(df.head())

Part 1: Rating Categorization.
This is the easiest part. An encoder

In [ ]:
save_path_drive = '/content/drive/MyDrive/593project/fine_tuned_bart_lora_classification_saved'

if os.path.exists(save_path_drive):
  try:
        base_model = BartForSequenceClassification.from_pretrained('facebook/bart-base')
  except Exception as e:
        print(f"An error occurred loading from Drive: {e}")
  loaded_peft_model = PeftModel.from_pretrained(base_model, save_path_drive)
  loaded_tokenizer = BartTokenizer.from_pretrained(save_path_drive)

# 1. Load the BART tokenizer and model for sequence classification
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# Prepare labels for classification
# Ensure 'Rating' column is present and convert to categorical codes
df['labels'] = df['Rating'].astype('category').cat.codes
num_labels = len(df['Rating'].astype('category').cat.categories)

model = BartForSequenceClassification.from_pretrained('facebook/bart-base', num_labels=num_labels)

# 2. Configure LoRA for Sequence Classification
lora_config = LoraConfig(
    r=8,  # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj"], # Target modules for BART's attention layers
    lora_dropout=0.1, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers ('none', 'all', or 'lora_only')
    task_type="SEQ_CLS" # Specify the task type for sequence classification
)

# 3. Get PEFT model
peft_model = get_peft_model(model, lora_config)
print("Trainable parameters after LoRA application:")
peft_model.print_trainable_parameters()

# Convert pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

def preprocess_function(examples):
    # Tokenize inputs (the text for classification)
    model_inputs = tokenizer(examples['text'], max_length=1024, truncation=True, padding="max_length")
    # Assign the numerical labels
    model_inputs["labels"] = examples["labels"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Split into training and evaluation sets (using existing small data for example)
# In a real scenario, you'd want proper splits using .train_test_split()
train_dataset = tokenized_dataset
eval_dataset = tokenized_dataset

In [ ]:
import torch
from torch.optim import AdamW
from transformers import get_scheduler

training_args = TrainingArguments(
    output_dir='./results_classification',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_classification',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=True, # Changed to True for bfloat16 mixed precision on XLA device (TPU)
    fp16=False, # Ensure fp16 is False when bf16 is True
    report_to='none',
    learning_rate=5e-5 # Added a default learning rate
)

# Define a custom optimizer to avoid the 'fused=True' error with XLA
# By default, torch.optim.AdamW does not use fused operations for XLA/CPU
optimizer = AdamW(peft_model.parameters(), lr=training_args.learning_rate)

# Calculate total training steps for the scheduler
total_train_steps = int(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs)

# Define a learning rate scheduler
lr_scheduler = get_scheduler(
    name=training_args.lr_scheduler_type, # Defaults to 'linear'
    optimizer=optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=total_train_steps,
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    optimizers=(optimizer, lr_scheduler) # Pass the custom optimizer and scheduler
)

trainer.train()

os.makedirs(save_path_drive, exist_ok=True)

In [ ]:
peft_model.to('cpu').save_pretrained(save_path_drive)
tokenizer.save_pretrained(save_path_drive)